In [1]:
# ============================================================================
# ENHANCED TRAINING AND PREDICTION PIPELINE FOR TELECOM CHURN
# ============================================================================
import pandas as pd
import numpy as np
import re
from datetime import datetime
from pathlib import Path
import sys

# Add project root to path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, QuantileTransformer
from sklearn.metrics import classification_report, accuracy_score, make_scorer, roc_auc_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, mutual_info_classif
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Import custom feature engineering utilities
from utils.feature_engineering import (
    AdvancedImputer,
    IntelligentFeatureSelector,
    TelecomFeatureEngineer,
)


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def clean_feature_names(columns):
    """Clean feature names to remove special characters for LightGBM compatibility"""
    cleaned = []
    for col in columns:
        cleaned_name = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
        cleaned_name = re.sub(r'_+', '_', cleaned_name)
        cleaned_name = cleaned_name.strip('_')
        cleaned.append(cleaned_name)
    return cleaned


def get_feature_columns(df):
    """Get the list of feature columns to use for modeling"""
    exclude_cols = {'Unnamed: 0', 'id', 'label', 'age_group', 'customer_segment'}
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    return feature_cols


def reduce_features_by_importance(X, y, model, threshold=0.001):
    """
    Reduce features based on feature importance
    """
    print(f"\nPerforming feature selection (threshold={threshold})...")
    selector = SelectFromModel(model, threshold=threshold, prefit=False)
    selector.fit(X, y)
    selected_features = X.columns[selector.get_support()].tolist()
    print(f"Selected {len(selected_features)} features out of {len(X.columns)}")
    return selected_features


# Define custom scoring function
def custom_score(y_true, y_pred, y_proba=None):
    """Calculate custom score: 0.7 * accuracy + 0.3 * f1"""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return 0.7 * acc + 0.3 * f1


custom_scorer = make_scorer(custom_score, needs_proba=False)


from sklearn.metrics import precision_recall_curve


def find_optimal_threshold(y_true, y_proba, metric='f1'):
    """Find optimal threshold for F1 score"""
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall)
    
    # Handle NaN values
    f1_scores = np.nan_to_num(f1_scores)
    
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    optimal_f1 = f1_scores[optimal_idx]
    
    return optimal_threshold, optimal_f1


# ============================================================================
# LOAD AND PREPARE TRAINING DATA
# ============================================================================
print("=" * 80)
print("LOADING TRAINING DATA")
print("=" * 80)

df_train = pd.read_csv('./train.csv')
print(f"Training data shape: {df_train.shape}")
print(f"\nClass distribution:")
print(df_train['label'].value_counts())
print(f"Class ratio: {df_train['label'].value_counts()[0] / df_train['label'].value_counts()[1]:.2f}:1")

# Check for missing values
print(f"\nMissing values summary:")
missing_summary = df_train.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if len(missing_summary) > 0:
    print(missing_summary.head(10))
else:
    print("No missing values found")

# ============================================================================
# ADVANCED FEATURE ENGINEERING WITH UTILITIES
# ============================================================================
print("\n" + "=" * 80)
print("CREATING ADVANCED FEATURES")
print("=" * 80)

# Step 1: Advanced imputation
print("Step 1: Performing advanced imputation...")
imputer = AdvancedImputer(strategy='auto')
df_train_imputed = imputer.fit_transform(df_train, target=df_train['label'])

# Step 2: Feature engineering
print("Step 2: Creating telecom-specific features...")
engineer = TelecomFeatureEngineer()
df_train_processed = engineer.create_all_features(
    df_train_imputed, 
    is_train=True, 
    target=df_train_imputed['label']
)

# Step 3: Get feature columns
exclude_cols = {'label', 'id', 'Unnamed: 0', 'age_group', 'customer_segment'}
feature_cols = [col for col in df_train_processed.columns if col not in exclude_cols]
print(f"Total features after engineering: {len(feature_cols)}")

# Prepare X and y
X = df_train_processed[feature_cols].copy()
y = df_train_processed['label']

# Handle remaining missing values and infinities
print("\nHandling missing values and infinities...")
X = X.replace([np.inf, -np.inf], 0)
X = X.fillna(0)

# Step 4: Intelligent feature selection
print("\nStep 4: Selecting most discriminative features...")
selector = IntelligentFeatureSelector(n_features=120)  # Keep top 120 features
selected_features = selector.select_features(X, y, method='ensemble')
X = X[selected_features]

print(f"Final feature count after selection: {X.shape[1]}")
print(f"\nTop 10 features by importance:")
for i, (feat, score) in enumerate(list(selector.feature_scores.items())[:10], 1):
    print(f"  {i}. {feat}: {score:.4f}")

# Clean feature names for compatibility
original_feature_cols = X.columns.tolist()
cleaned_feature_cols = clean_feature_names(original_feature_cols)
feature_name_mapping = dict(zip(cleaned_feature_cols, original_feature_cols))
X.columns = cleaned_feature_cols

print(f"\nFinal feature matrix shape: {X.shape}")
print(f"Feature statistics:")
print(f"  Numeric features: {X.select_dtypes(include=[np.number]).shape[1]}")
print(f"  Memory usage: {X.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================================================
# FEATURE SCALING (Optional - for some models)
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE SCALING")
print("=" * 80)

# Create scaled version for models that benefit from it
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)
print("Features scaled using StandardScaler")

# ============================================================================
# TRAIN-VALIDATION SPLIT
# ============================================================================
print("\n" + "=" * 80)
print("SPLITTING DATA FOR VALIDATION")
print("=" * 80)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.05, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Training class distribution: {y_train.value_counts().to_dict()}")
print(f"Validation class distribution: {y_val.value_counts().to_dict()}")

LOADING TRAINING DATA
Training data shape: (59904, 88)

Class distribution:
label
0    44904
1    15000
Name: count, dtype: int64
Class ratio: 2.99:1

Missing values summary:
gm_use_dur               47451
gm_dayt_use_dur          47451
gm_ngt_use_dur           47451
shrt_vid_use_dur         47451
shrt_vid_dayt_use_dur    47451
shrt_vid_ngt_use_dur     47451
long_vid_use_dur         47451
long_vid_dayt_use_dur    47451
long_vid_ngt_use_dur     47451
anchor_use_dur           47451
dtype: int64

CREATING ADVANCED FEATURES
Step 1: Performing advanced imputation...
Step 2: Creating telecom-specific features...

ADVANCED FEATURE ENGINEERING
Creating usage pattern features...
Creating temporal features...
Creating behavioral segments...
Creating revenue features...
Creating engagement metrics...
Creating risk indicators...
Creating interaction features...
Creating statistical features...
Creating target-encoded features...
Creating polynomial features...
Creating aggregation features...
Appl

In [2]:

# ============================================================================
# ADVANCED MODEL TRAINING WITH CROSS-VALIDATION
# ============================================================================
print("\n" + "=" * 80)
print("TRAINING ADVANCED MODELS WITH CROSS-VALIDATION")
print("=" * 80)

# Calculate scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Setup cross-validation
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Store results
results = {}



TRAINING ADVANCED MODELS WITH CROSS-VALIDATION
Scale pos weight: 2.99


In [3]:
# ============================================================================
# MODEL 2: XGBOOST WITH ADVANCED PARAMETERS
# ============================================================================
print("\n" + "-" * 80)
print("2. TRAINING XGBOOST")
print("-" * 80)

xgb_params_enhanced = {
    'n_estimators': 1225,
    'max_depth': 9,
    'learning_rate': 0.02,
    'subsample': 0.85,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,
    'colsample_bynode': 0.8,
    'min_child_weight': 3,
    'gamma': 0.05,
    'reg_alpha': 0.4,
    'reg_lambda': 0.4,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'tree_method': 'hist',
    'grow_policy': 'depthwise',
    'max_bin': 512,          # Increased bins
    # 'sampling_method': 'gradient_based',
    'eval_metric': 'auc'
}

xgb_model = xgb.XGBClassifier(**xgb_params_enhanced)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_xgb = cross_val_score(xgb_model, X_train, y_train, cv=skf, 
#                                  scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_xgb}")
# print(f"Mean CV ROC-AUC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")

# Train on full training set with early stopping
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    # early_stopping_rounds=50,
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_val)
y_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]

xgb_acc = accuracy_score(y_val, y_pred_xgb)
xgb_auc = roc_auc_score(y_val, y_proba_xgb)
xgb_f1 = f1_score(y_val, y_pred_xgb)
xgb_custom_score = custom_score(y_val, y_pred_xgb, y_proba_xgb)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {xgb_acc:.4f}")
print(f"  ROC-AUC: {xgb_auc:.4f}")
print(f"  F1-Score: {xgb_f1:.4f}")
print(f"  Custom Score: {xgb_custom_score:.4f}")

results['XGBoost'] = {
    'model': xgb_model,
    # 'cv_scores': cv_scores_xgb,
    'val_auc': xgb_auc,
    'val_f1': xgb_f1,
    'predictions': y_pred_xgb,
    'probabilities': y_proba_xgb,
    'custom_score': xgb_custom_score
    
}


--------------------------------------------------------------------------------
2. TRAINING XGBOOST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.8061
  ROC-AUC: 0.8461
  F1-Score: 0.6366
  Custom Score: 0.7552


In [4]:

# ============================================================================
# MODEL COMPARISON AND SELECTION
# ============================================================================
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)

# Create comparison dataframe
comparison_data = []
for model_name, model_results in results.items():
    if 'cv_scores' in model_results:
        cv_mean = model_results['cv_scores'].mean()
        cv_std = model_results['cv_scores'].std()
    else:
        cv_mean = cv_std = np.nan
    
    comparison_data.append({
        'Model': model_name,
        'CV_AUC_Mean': cv_mean,
        'CV_AUC_Std': cv_std,
        'Val_AUC': model_results['val_auc'],
        'Val_F1': model_results['val_f1'],
        'Custom_Score': model_results['custom_score']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Custom_Score', ascending=False)

print("\nModel Performance Comparison:")
print("=" * 60)
for _, row in comparison_df.iterrows():
    # if pd.isna(row['CV_AUC_Mean']):
    #     print(f"{row['Model']:20s} | Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f}")
    # else:
    #     print(f"{row['Model']:20s} | CV AUC: {row['CV_AUC_Mean']:.4f}±{row['CV_AUC_Std']:.4f} | "
    #           f"Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f}")
    print(f"{row['Model']:20s} | Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f} | "
          f"Custom Score: {row['Custom_Score']:.4f}")

# Select best model
best_model_name = comparison_df.iloc[0]['Model']
best_model_results = results[best_model_name]
# best_model_name = 'VotingClassifier'
# best_model_results = results['VotingClassifier']


print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"   Validation F1:  {best_model_results['val_f1']:.4f}")
print(f"   Custom Score:   {best_model_results['custom_score']:.4f}")

# ============================================================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

def get_feature_importance(model, feature_names, model_type):
    """Extract feature importance from different model types"""
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importance = abs(model.coef_[0])
    else:
        return None
    
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    return feature_importance

# Get feature importance from best tree-based models
models_for_importance = ['LightGBM', 'XGBoost', 'CatBoost', 'RandomForest']

print("Top 20 Most Important Features:")
print("-" * 50)

for model_name in models_for_importance:
    if model_name in results and 'model' in results[model_name]:
        importance_df = get_feature_importance(
            results[model_name]['model'], 
            X.columns, 
            model_name
        )
        
        if importance_df is not None:
            print(f"\n{model_name}:")
            top_features = importance_df.head(10)
            for idx, row in top_features.iterrows():
                original_name = feature_name_mapping.get(row['feature'], row['feature'])
                print(f"  {row['feature'][:30]:30s} | {row['importance']:.4f}")

# ============================================================================
# SAVE MODELS AND RESULTS
# ============================================================================
print("\n" + "=" * 80)
print("SAVING MODELS AND RESULTS")
print("=" * 80)

import pickle
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# Save best model
best_model = best_model_results['model'] if 'model' in best_model_results else None
if best_model is not None:
    with open(f'models/best_model_{best_model_name.lower()}.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    print(f"✅ Best model ({best_model_name}) saved to models/")

# Save feature columns and preprocessing objects
with open('models/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

with open('models/feature_name_mapping.pkl', 'wb') as f:
    pickle.dump(feature_name_mapping, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save results summary
results_summary = {
    'model_comparison': comparison_df.to_dict('records'),
    'best_model_name': best_model_name,
    'best_model_auc': best_model_results['val_auc'],
    'best_model_f1': best_model_results['val_f1'],
    'custom_score': best_model_results['custom_score'],
    'feature_count': len(feature_cols),
    'training_samples': len(X_train),
    'validation_samples': len(X_val)
}

with open('models/training_results.pkl', 'wb') as f:
    pickle.dump(results_summary, f)

print(f"✅ Feature preprocessing objects saved")
print(f"✅ Training results summary saved")

print("\n" + "=" * 80)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"🎯 Best Model: {best_model_name}")
print(f"📊 Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"📈 Validation F1: {best_model_results['val_f1']:.4f}")
print(f"🧮 Total Features Used: {len(feature_cols)}")
print(f"💾 Models saved to 'models/' directory")
print("=" * 80)


MODEL COMPARISON SUMMARY

Model Performance Comparison:
XGBoost              | Val AUC: 0.8461 | Val F1: 0.6366 | Custom Score: 0.7552

🏆 BEST MODEL: XGBoost
   Validation AUC: 0.8461
   Validation F1:  0.6366
   Custom Score:   0.7552

FEATURE IMPORTANCE ANALYSIS
Top 20 Most Important Features:
--------------------------------------------------

XGBoost:
  if_nulim_prod                  | 0.1023
  arpu_sqrt                      | 0.0437
  arpu                           | 0.0427
  annual_revenue_potential       | 0.0405
  arpu_tenure_ratio              | 0.0363
  monthly_arpu                   | 0.0300
  cm_flux_tot_cnt                | 0.0272
  billing_trend                  | 0.0119
  has_overusage                  | 0.0099
  high_overage_risk              | 0.0097

SAVING MODELS AND RESULTS
✅ Best model (XGBoost) saved to models/
✅ Feature preprocessing objects saved
✅ Training results summary saved

TRAINING COMPLETED SUCCESSFULLY!
🎯 Best Model: XGBoost
📊 Validation AUC: 0.8461
📈 

In [5]:
# ============================================================================
# LOAD AND PREDICT TEST DATA
# ============================================================================

print("\n" + "=" * 80)
print("LOADING TEST DATA")
print("=" * 80)

df_test = pd.read_csv('./testB.csv')
print(f"Test data shape: {df_test.shape}")

# Store IDs
test_ids = df_test['id'].copy()

# ============================================================================
# APPLY SAME PREPROCESSING PIPELINE TO TEST DATA
# ============================================================================
print("\n" + "=" * 80)
print("PREPROCESSING TEST DATA")
print("=" * 80)

# Step 1: Apply the same imputation (using transform, not fit_transform)
print("Step 1: Applying imputation to test data...")
df_test_imputed = imputer.transform(df_test)

# Step 2: Apply the same feature engineering (using is_train=False)
print("Step 2: Creating features for test data...")
df_test_processed = engineer.create_all_features(
    df_test_imputed, 
    is_train=False, 
    target=None
)

# Step 3: Align features with training data
print("Step 3: Aligning test features with training data...")
# Use reindex to ensure we have the exact same features in the same order
X_test_full = df_test_processed.reindex(columns=feature_cols)

# Handle remaining missing values and infinities
X_test_full = X_test_full.replace([np.inf, -np.inf], 0)
X_test_full = X_test_full.fillna(0)

# Step 4: Select only the features that were selected during training
print("Step 4: Selecting trained features...")
X_test = X_test_full[selected_features].copy()

# Clean feature names to match training
X_test.columns = cleaned_feature_cols

# Check for feature alignment issues
missing_features = [col for col in selected_features if col not in df_test_processed.columns]
if missing_features:
    print(f"\nWarning: {len(missing_features)} features missing in test data (filled with zeros)")
    print(f"  Sample missing features: {missing_features[:5]}")
else:
    print("✓ All training features present in test data")

print(f"\nTest feature matrix shape: {X_test.shape}")
print(f"Training feature matrix shape: {X_train.shape}")

# Verify feature alignment
if X_test.shape[1] != X_train.shape[1]:
    print(f"\n⚠️  Warning: Feature count mismatch!")
    print(f"  Training: {X_train.shape[1]} features")
    print(f"  Test: {X_test.shape[1]} features")
else:
    print("✓ Feature counts match between train and test")

# Verify column names match
if list(X_test.columns) != list(X_train.columns):
    print("\n⚠️  Warning: Feature names don't match exactly")
    # Find differences
    train_only = set(X_train.columns) - set(X_test.columns)
    test_only = set(X_test.columns) - set(X_train.columns)
    if train_only:
        print(f"  Features only in training: {list(train_only)[:5]}")
    if test_only:
        print(f"  Features only in test: {list(test_only)[:5]}")
else:
    print("✓ Feature names match exactly between train and test")

# ============================================================================
# MAKE PREDICTIONS
# ============================================================================

print("\n" + "=" * 80)
print("MAKING PREDICTIONS")
print("=" * 80)

# Get predictions from best model
predictions = best_model.predict(X_test)
predictions_proba = best_model.predict_proba(X_test)[:, 1] if len(best_model.predict_proba(X_test).shape) >= 2 else best_model.predict_proba(X_test)

print(f"\nPredictions distribution:")
print(f"  Class 0: {(predictions == 0).sum():,} ({(predictions == 0).sum() / len(predictions) * 100:.1f}%)")
print(f"  Class 1: {(predictions == 1).sum():,} ({(predictions == 1).sum() / len(predictions) * 100:.1f}%)")

print(f"\nProbability statistics:")
print(f"  Mean probability: {predictions_proba.mean():.4f}")
print(f"  Median probability: {np.median(predictions_proba):.4f}")
print(f"  Std probability: {predictions_proba.std():.4f}")
print(f"  Min probability: {predictions_proba.min():.4f}")
print(f"  Max probability: {predictions_proba.max():.4f}")

# ============================================================================
# SAVE RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'label': predictions
})

# Save to CSV
output_path = './submit.csv'
submission.to_csv(output_path, index=False)

print(f"✅ Submission saved to: {output_path}")
print(f"   Submission shape: {submission.shape}")
print(f"\nFirst 10 predictions:")
print(submission.head(10))

# Also save with probabilities for analysis
submission_with_proba = pd.DataFrame({
    'id': test_ids,
    'label': predictions,
    'probability': predictions_proba
})

output_path_proba = './submit_with_probabilities.csv'
submission_with_proba.to_csv(output_path_proba, index=False)
print(f"\n✅ Predictions with probabilities saved to: {output_path_proba}")

# ============================================================================
# SAVE PREPROCESSING OBJECTS FOR LATER USE
# ============================================================================
print("\n" + "=" * 80)
print("SAVING PREPROCESSING PIPELINE")
print("=" * 80)

# Save the imputer and engineer for future use
with open('models/imputer.pkl', 'wb') as f:
    pickle.dump(imputer, f)

with open('models/engineer.pkl', 'wb') as f:
    pickle.dump(engineer, f)

with open('models/selected_features.pkl', 'wb') as f:
    pickle.dump(selected_features, f)

print("✅ Preprocessing pipeline saved:")
print("   - models/imputer.pkl")
print("   - models/engineer.pkl")
print("   - models/selected_features.pkl")

print("\n" + "=" * 80)
print("PIPELINE COMPLETE!")
print("=" * 80)
print(f"🎯 Best Model: {best_model_name}")
print(f"📊 Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"📈 Validation F1: {best_model_results['val_f1']:.4f}")
print(f"🧮 Features Used: {X_test.shape[1]}")
print(f"🔮 Predictions Made: {len(predictions):,}")
print(f"💾 Files saved: submit.csv, submit_with_probabilities.csv")
print("=" * 80)


LOADING TEST DATA
Test data shape: (19999, 87)

PREPROCESSING TEST DATA
Step 1: Applying imputation to test data...
Step 2: Creating features for test data...

ADVANCED FEATURE ENGINEERING
Creating usage pattern features...
Creating temporal features...
Creating behavioral segments...
Creating revenue features...
Creating engagement metrics...
Creating risk indicators...
Creating interaction features...
Creating statistical features...
Creating target-encoded features...
Creating polynomial features...
Creating aggregation features...
Applying mathematical transformations...

✓ Feature engineering complete. Total features: 351
Step 3: Aligning test features with training data...
Step 4: Selecting trained features...

  Sample missing features: ['is_data_centric_log1p']

Test feature matrix shape: (19999, 120)
Training feature matrix shape: (56908, 120)
✓ Feature counts match between train and test
✓ Feature names match exactly between train and test

MAKING PREDICTIONS

Predictions di